# AlignTune SFT

SFT is the method that worked in the quick evaluation. This kernel trains the same adapter
through AlignTune's `create_sft_trainer`, so the headline model comes from the Track 1
post-training tool. The DPO path through AlignTune is kept as a negative result.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits=[p for p in Path("/kaggle/input").rglob("rule_checker.py") if p.exists() and (p.parents[2]/"scripts"/"train.py").exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits: raise SystemExit("code not mounted")
    return hits[0].parents[2]

WORK=Path("/kaggle/working/htd")
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(find_repo(), WORK)
os.chdir(WORK); sys.path.insert(0,str(WORK/"src"))
for p in Path("/kaggle/input").rglob("hf_token.txt"):
    os.environ.setdefault("HF_TOKEN", p.read_text().strip())
    os.environ.setdefault("HUGGINGFACE_API_KEY", p.read_text().strip())
assert os.environ.get("HF_TOKEN")
SFT=next(Path("/kaggle/input").rglob("sft_train.jsonl"))
print("SFT data:", SFT)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout)

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def assert_adapter(name):
    p = WORK / "outputs" / name
    cfg = p / "adapter_config.json"
    weights = p / "adapter_model.safetensors"
    assert cfg.exists(), f"{name}: adapter_config.json missing"
    assert weights.exists() and weights.stat().st_size > 1_000_000, f"{name}: adapter weights missing or empty"
    print(f"{name}: adapter ok")


In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "aligntune"])

In [ ]:
run([sys.executable, "scripts/train_aligntune.py", "--method", "sft", "--data", str(SFT),
     "--output", "outputs/aligntune_sft_smoke", "--max-steps", "10", "--grad-accum", "2"])
assert_adapter("aligntune_sft_smoke")

In [ ]:
run([sys.executable, "scripts/train_aligntune.py", "--method", "sft", "--data", str(SFT),
     "--output", "outputs/aligntune_sft", "--epochs", "1"])
assert_adapter("aligntune_sft")

In [ ]:
for name in ("aligntune_sft_smoke","aligntune_sft"):
    p=WORK/"outputs"/name
    print(name, sorted(x.name for x in p.iterdir()) if p.exists() else "MISSING")